# L22 · RAG：给 AI 外接大脑

**学习目标**
- 理解 RAG（检索增强生成）：先查资料，再回答
- 理解「向量化 + 相似度检索」如何让机器「懂」语义
- 亲手实现一个迷你 RAG 检索器（离线，无需 LLM API）

**前置依赖**：L19（Embedding 概念）、L08 pandas、L21  
**预计时长**：50 分钟  
**技术栈**：`scikit-learn`、`numpy`（TF-IDF 向量化，离线可运行）

---

## 概念讲解：RAG = 开卷考试

大模型训练完就「定格」了，不知道你公司的内部文档。
**RAG** 让 AI 像「开卷考试」：回答前，先去你的资料库里**检索最相关的几段**，喂给模型再生成答案。

核心两步：
1. **索引**：把文档切成块，每块变成向量（数字指纹）存起来
2. **检索**：把问题也变向量，找「指纹最像」的文档块

本课我们用 TF-IDF（一种词频向量）实现语义检索，零外部依赖。

## 第一步：建知识库 + 向量化索引

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

docs = [
    "我们的年假政策是入职满一年可享 10 天带薪年假。",
    "报销需要在消费后 30 天内提交发票，逾期作废。",
    "公司食堂午餐 12 元，晚餐 15 元，周末不供应。",
    "新电脑申请需部门主管审批，平均 3 个工作日到账。",
    "弹性工作时间为 10:00-16:00 必须在岗，其余自选。",
]
vectorizer = TfidfVectorizer()
doc_vecs = vectorizer.fit_transform(docs)
print("知识库已建好，共", len(docs), "条政策片段。")

## 第二步：检索最相关片段

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def retrieve(question, top_k=2):
    q_vec = vectorizer.transform([question])
    sims = cosine_similarity(q_vec, doc_vecs).flatten()
    idx = np.argsort(-sims)[:top_k]
    return [(docs[i], round(sims[i], 3)) for i in idx]

for q in ["年假怎么算", "吃饭多少钱", "电脑怎么申请"]:
    hits = retrieve(q)
    print(f"\n问：{q}")
    for text, score in hits:
        print(f"  相似度 {score}：{text}")

# 🎯 AHA 顿悟单元格：你的「公司政策问答机器人」

运行下面代码。你会得到一个**能回答私有知识的检索器**：输入任何关于公司政策的问题，
它从知识库里找出最相关的原文片段，并给出相似度。改 `docs` 换成你自己的资料也能用。

> 这就是 ChatPDF、企业知识库、客服机器人的核心。你刚搭的检索器，和那些收费产品用的是同一种原理——
> 只不过它们用更强的向量模型、更大的库。骨架，你已经完全掌握了。

In [ ]:
# ===== 运行我！输入你的问题，看它从知识库检索 =====
questions = [
    "我想请年假，有什么规定？",
    "报销发票最晚什么时候交？",
    "食堂晚上还开吗？",
    "什么时候必须在公司？",
]
print("  📚 公司政策问答机器人（RAG 检索器）已就绪\n")
for q in questions:
    hits = retrieve(q, top_k=1)
    text, score = hits[0]
    print(f"  👤 问：{q}")
    print(f"  🔍 检索到（相似度 {score}）：{text}")
    print(f"  🤖 答：根据资料，「{text}」\n")
print("  ✨ 它能回答训练时从未见过的私有知识 —— 这就是 RAG 外接大脑的力量！")

# 📝 讲师备课笔记（接手 Agent 专用）

**本课难点**：向量/相似度直觉；TF-IDF 作为「简化版 embedding」需说明真实 RAG 用神经网络 embedding（如 OpenAI text-embedding）。  
**易错点**：`cosine_similarity` 输入需 2D；中文分词（TF-IDF 默认按字符，够用但提一句 jieba 更佳）。  
**AHA 机制**：私有知识问答，强「AI 懂我的文档」实感，离线可跑（关键：无 API key 依赖）。  
**衔接**：L23 Agent（RAG 作为工具之一）；L25 评测（评测 RAG 答案质量）；L37 综合项目。  
**真 LLM 衔接**：注明真实 RAG = 检索到的片段 + 问题 拼成 prompt 送给 LLM 生成自然语言答案，本方案因离线用「原文回显」代替生成。  
**依赖**：`pip install scikit-learn numpy`。

# 📚 作业 / 下一步

1. 把 `docs` 换成你自己的 5 条笔记，再问问题。
2. 把 `top_k` 改成 3，看是否召回更多片段。
3. 下一课 **L23 AI Agent：会自己规划的助手** —— 让 AI 把多个工具串成一条任务链。